In [ ]:
#
# Cell 1A: mount Google Drive and define file paths
#
import os
import json
import time
from google.colab import drive

# Connect Colab to Google Drive
drive.mount('/content/drive')

# --------Folder paths--------
BASE_DIR = '/content/drive/MyDrive/DSA_Search_Engine' #the project folder.
INDEX_DIR = os.path.join(BASE_DIR, 'index_data')   # all index outputs go here

# --- Input / output file paths ---
DATASET_PATH = os.path.join(BASE_DIR, 'cleaned_dataset.jsonl')   # full 55,000 reviews
SAMPLE_PATH  = os.path.join(BASE_DIR, 'sample_dataset.jsonl')    # 400-review test file

# Create index_data/ if it doesn't exist yet
os.makedirs(INDEX_DIR, exist_ok=True)

# ---------- Quick check that everything is where we expect------------
print("Dataset found :", os.path.exists(DATASET_PATH))
print("index_data/   :", os.path.exists(INDEX_DIR))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Dataset found : True
index_data/   : True


In [ ]:
import sys

if BASE_DIR not in sys.path:
    sys.path.insert(0, BASE_DIR)

from src.preprocessing import tokenize as module_tokenize

test_texts = [
    "The battery life is GREAT, but the controller drifts!",
    "I loved this game and its graphics.",
    "Don't buy this product. It wasn't worth it.",
    "123 players enjoyed the multiplayer experience."
]

for text in test_texts:
    original = tokenize(text)
    new = module_tokenize(text)

    print("Original:", original)
    print("New:     ", new)
    print("Match:", original == new)
    print()

    assert original == new

print("SUCCESS: Module tokenizer matches the original tokenizer.")

Original: ['the', 'batteri', 'life', 'is', 'great', 'but', 'the', 'control', 'drift']
New:      ['the', 'batteri', 'life', 'is', 'great', 'but', 'the', 'control', 'drift']
Match: True

Original: ['i', 'love', 'thi', 'game', 'and', 'it', 'graphic']
New:      ['i', 'love', 'thi', 'game', 'and', 'it', 'graphic']
Match: True

Original: ["don't", 'buy', 'thi', 'product', 'it', "wasn't", 'worth', 'it']
New:      ["don't", 'buy', 'thi', 'product', 'it', "wasn't", 'worth', 'it']
Match: True

Original: ['123', 'player', 'enjoy', 'the', 'multiplay', 'experi']
New:      ['123', 'player', 'enjoy', 'the', 'multiplay', 'experi']
Match: True

SUCCESS: Module tokenizer matches the original tokenizer.


In [ ]:
from src.preprocessing import (
    normalize_record as module_normalize_record,
    analyze_document as module_analyze_document
)

test_record = {
    "doc_id": 999,
    "title": "Great Battery Performance",
    "text": "The battery life is great. The battery lasts long.",
    "rating": 4.5,
    "helpful_vote": 12,
    "timestamp": 1700000000000
}

# Test record normalization
original_record = normalize_record(test_record)
new_record = module_normalize_record(test_record)

print("Original record:", original_record)
print("New record:     ", new_record)
print("Match:", original_record == new_record)

assert original_record == new_record

# Test document analysis
original_stats, original_length = analyze_document(original_record)
new_stats, new_length = module_analyze_document(new_record)

print("\nOriginal analysis:", original_stats)
print("New analysis:     ", new_stats)

print("\nOriginal document length:", original_length)
print("New document length:     ", new_length)

assert original_stats == new_stats
assert original_length == new_length

print("\nSUCCESS: Both functions match the original notebook.")

Original record: {'doc_id': 999, 'title': 'Great Battery Performance', 'text': 'The battery life is great. The battery lasts long.', 'rating': 4.5, 'helpful_vote': 12, 'timestamp': 1700000000000}
New record:      {'doc_id': 999, 'title': 'Great Battery Performance', 'text': 'The battery life is great. The battery lasts long.', 'rating': 4.5, 'helpful_vote': 12, 'timestamp': 1700000000000}
Match: True

Original analysis: {'the': [2, 0, 0], 'batteri': [2, 1, 1], 'life': [1, 0, 2], 'is': [1, 0, 3], 'great': [1, 1, 4], 'last': [1, 0, 7], 'long': [1, 0, 8], 'perform': [0, 1, -1]}
New analysis:      {'the': [2, 0, 0], 'batteri': [2, 1, 1], 'life': [1, 0, 2], 'is': [1, 0, 3], 'great': [1, 1, 4], 'last': [1, 0, 7], 'long': [1, 0, 8], 'perform': [0, 1, -1]}

Original document length: 9
New document length:      9

SUCCESS: Both functions match the original notebook.


In [ ]:
from src.preprocessing import iter_records as module_iter_records

# Read all sample records using both implementations
original_records = list(iter_records(SAMPLE_PATH))
new_records = list(module_iter_records(SAMPLE_PATH))

# Compare results
print("Original records:", len(original_records))
print("New records:     ", len(new_records))

print("First record matches:",
      original_records[0] == new_records[0])

print("Last record matches:",
      original_records[-1] == new_records[-1])

print("All records match:",
      original_records == new_records)

assert original_records == new_records

print("\nSUCCESS: iter_records() matches the original notebook.")

Original records: 400
New records:      400
First record matches: True
Last record matches: True
All records match: True

SUCCESS: iter_records() matches the original notebook.


In [ ]:
#
# Cell 1B: Tokenizer : cuts a piece of text into a clean list of words that the indexes can use.
# LOWERCASE each word
# trims extra appostrophies
#Stemming: coverts words to rooot words, eg drifting to drift

import re # regex: we use ot to find words
from functools import lru_cache # its a memoery, once we have found smth, it will remeber it, reduces time
import nltk # human lang library for stemming and stopwords
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer # rules for stemming, words-> roots

nltk.download('stopwords', quiet=True) # to download it and quiet if ale exists

STOPWORDS = set(stopwords.words('english')) # returns list of eng stopwords
_stemmer = PorterStemmer()

@lru_cache(maxsize=None)          # each unique word is stemmed only once
def stem_word(word):              # helps lru_cache to remember words
    return _stemmer.stem(word)

WORD_PATTERN = re.compile(r"[a-z0-9']+")   # a run of letters/digits/apostrophes

def tokenize(text, remove_stopwords=False):
    """Return a list of stemmed lowercase tokens.
    Stopwords are KEPT by default so 'data structures and algorithms'
    can match on 'and' (spec Requirement 5)."""
    tokens = []
    for w in WORD_PATTERN.findall(text.lower()):
        w = w.strip("'")
        if not w:
            continue
        if remove_stopwords and w in STOPWORDS:
            continue
        tokens.append(stem_word(w))
    return tokens

print(tokenize("The battery life is GREAT, but the controller drifts!"))

['the', 'batteri', 'life', 'is', 'great', 'but', 'the', 'control', 'drift']


In [ ]:
#
# Cell 1C: Create a 400-review sample file for testing
#
SAMPLE_SIZE = 400

count = 0
# Read the big file one line at a time; write only the first 400 lines
# r means read, utf-8 is for handling special characters
# w means write
with open(DATASET_PATH, 'r', encoding='utf-8') as src, \
     open(SAMPLE_PATH, 'w', encoding='utf-8') as dst:

    for line in src:  # to write line by line from src to dst
        dst.write(line)
        count += 1
        if count >= SAMPLE_SIZE:
            break

print(f"Sample created with {count} reviews") # count no of reviews stored
print(f"Saved at: {SAMPLE_PATH}")

#  Check: read it back, count the lines, inspect one timestamp ---
with open(SAMPLE_PATH, 'r', encoding='utf-8') as f:
    records = [json.loads(line) for line in f]

print(f"Lines in sample file: {len(records)}")
print(f"First doc_id: {records[0]['doc_id']} | Last doc_id: {records[-1]['doc_id']}")
print(f"Example timestamp: {records[0]['timestamp']}")

Sample created with 400 reviews
Saved at: /content/drive/MyDrive/DSA_Search_Engine/sample_dataset.jsonl
Lines in sample file: 400
First doc_id: 0 | Last doc_id: 399
Example timestamp: 1602718512453


In [ ]:
# for the extra dataset (handling ,0)
# Cell 2A: Settings + normalize_record (safe defaults)
#
USE_SAMPLE = False     # True = 400-review test run, False = full 55,000
REBUILD = False

SOURCE_PATH = SAMPLE_PATH if USE_SAMPLE else DATASET_PATH
OUT_DIR = os.path.join(INDEX_DIR, 'sample') if USE_SAMPLE else INDEX_DIR
os.makedirs(OUT_DIR, exist_ok=True)

LEXICON_PATH  = os.path.join(OUT_DIR, 'lexicon.json')
FORWARD_PATH  = os.path.join(OUT_DIR, 'forward_index.jsonl')
INVERTED_PATH = os.path.join(OUT_DIR, 'inverted_index.json')
META_PATH     = os.path.join(OUT_DIR, 'index_meta.json')

DEFAULT_RATING = 3.0   # neutral value: 0 would sit below the real 1-5 range


def _to_number(value, cast, default):
    """Convert value using cast (int or float).
    Returns default if value is None or cannot be converted."""
    if value is None:
        return default
    try:
        return cast(value)
    except (TypeError, ValueError):
        return default


def normalize_record(raw, doc_id=None):
    """Turn a raw dict into a clean record with all six fields.
    Raises ValueError if the record has no usable text or no doc_id."""
    if not isinstance(raw, dict):
        raise ValueError("record is not a dict")

    # text is the only truly required field
    text = raw.get('text')
    text = text.strip() if isinstance(text, str) else ''
    if not text:
        raise ValueError("record has no text")

    title = raw.get('title')
    title = title.strip() if isinstance(title, str) else ''

    # doc_id: use the one passed in, otherwise the one in the record
    if doc_id is None:
        doc_id = _to_number(raw.get('doc_id'), int, None)
    if doc_id is None:
        raise ValueError("record has no doc_id")

    rating = _to_number(raw.get('rating'), float, DEFAULT_RATING)
    rating = min(5.0, max(1.0, rating))            # keep inside 1 to 5

    votes = _to_number(raw.get('helpful_vote'), int, 0)
    votes = max(0, votes)                           # never negative

    now_ms = int(time.time() * 1000)                # Amazon timestamps are in milliseconds
    timestamp = _to_number(raw.get('timestamp'), int, now_ms)

    return {
        "doc_id": doc_id,
        "title": title,
        "text": text,
        "rating": rating,
        "helpful_vote": votes,
        "timestamp": timestamp,
    }


# --- Tests: broken records must be fixed or rejected, never crash ---
print(normalize_record({"text": "Great game", "rating": None,
                        "helpful_vote": None, "timestamp": None}, doc_id=1))
print(normalize_record({"text": "Fine", "rating": "9", "helpful_vote": -4}, doc_id=2))

try:
    normalize_record({"text": "   "}, doc_id=3)
except ValueError as e:
    print("Rejected empty review:", e)

{'doc_id': 1, 'title': '', 'text': 'Great game', 'rating': 3.0, 'helpful_vote': 0, 'timestamp': 1795574874468}
{'doc_id': 2, 'title': '', 'text': 'Fine', 'rating': 5.0, 'helpful_vote': 0, 'timestamp': 1795574874468}
Rejected empty review: record has no text


In [ ]:
#
# Cell 2B: analyze_document + iter_records
#
def analyze_document(record):
    """Return (stats, doc_len) for one clean record.
    stats maps each term to [tf, in_title, first_pos]:
      tf        = how many times the term appears in the text
      in_title  = 1 if the term also appears in the title, else 0
      first_pos = index of the first occurrence in the text (-1 if title-only)
    doc_len = number of tokens in the text."""
    text_tokens = tokenize(record['text'])

    stats = {}
    for pos, term in enumerate(text_tokens):
        if term in stats:
            stats[term][0] += 1              # seen before: just add to tf
        else:
            stats[term] = [1, 0, pos]        # first time: tf=1, remember position

    # Title terms: use a set so a repeated title word is counted once
    for term in set(tokenize(record['title'])):
        if term in stats:
            stats[term][1] = 1               # word is in text and in title
        else:
            stats[term] = [0, 1, -1]         # title-only word

    return stats, len(text_tokens)


def iter_records(path):
    """Yield clean records from a JSONL file, one at a time.
    Bad lines (invalid JSON, no text, no doc_id) are skipped and counted."""
    skipped = 0
    with open(path, 'r', encoding='utf-8') as f:
        for line_no, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue                     # ignore blank lines
            try:
                yield normalize_record(json.loads(line))
            except ValueError as e:          # bad JSON is also a ValueError
                skipped += 1
                if skipped <= 5:
                    print(f"Skipped line {line_no}: {e}")
    if skipped:
        print(f"Total skipped lines: {skipped}")


# --- Test 1: analyze one small document by hand ---
stats, n = analyze_document({"title": "Great battery",
                             "text": "The battery lasts long. Battery good."})
print("doc_len:", n)
print(stats)

# --- Test 2: stream the sample file ---
count = 0
for rec in iter_records(SOURCE_PATH):
    count += 1
print("Records read from source:", count)

doc_len: 6
{'the': [1, 0, 0], 'batteri': [2, 1, 1], 'last': [1, 0, 2], 'long': [1, 0, 3], 'good': [1, 0, 5], 'great': [0, 1, -1]}
Records read from source: 55000


In [ ]:
# Cell 2C: lexicon
# format: {word: [id, df]}
# id = number given to the word, df = number of reviews that contain the word

import psutil
from collections import Counter


def memory_mb():
    # memory used by this notebook right now, in MB
    return psutil.Process(os.getpid()).memory_info().rss / 1e6


def load_lexicon(path):
    # read a saved lexicon file
    with open(path, 'r', encoding='utf-8') as f:
        lexicon = json.load(f)

    # the old format stored dicts, the new one stores lists
    first = next(iter(lexicon.values()), None)
    if first is not None and ( not isinstance(first, list) or len(first) != 2):
        raise ValueError("lexicon.json has the old format, set REBUILD = True in cell 1A")
    return lexicon


def build_lexicon(source_path, lexicon_path):
    # read every review and count in how many reviews each word appears (df)
    df = Counter()
    for record in iter_records(source_path):
        stats, _ = analyze_document(record)    # stats has one key per unique word
        df.update(stats.keys())                # each word counts once per review

    # sorted words get ids 0, 1, 2... so the ids are the same on every run
    lexicon = {term: [i, df[term]] for i, term in enumerate(sorted(df))}

    with open(lexicon_path, 'w', encoding='utf-8') as f:
        json.dump(lexicon, f)
    return lexicon


start_mem = memory_mb()
t0 = time.time()

if os.path.exists(LEXICON_PATH) and not REBUILD:
    lexicon = load_lexicon(LEXICON_PATH)
    action = "loaded"
else:
    lexicon = build_lexicon(SOURCE_PATH, LEXICON_PATH)
    action = "built"

print(f"Lexicon {action} in {time.time() - t0:.1f}s")
print(f"Unique words : {len(lexicon):,}")
print(f"Memory       : {memory_mb():.0f} MB now (+{memory_mb() - start_mem:.0f} MB for the lexicon)")
print(f"File         : {LEXICON_PATH} ({os.path.getsize(LEXICON_PATH) / 1e6:.2f} MB)")
print("First 3      :", {w: lexicon[w] for w in list(lexicon)[:3]})

# most common words, sorted by df (second number)
top = sorted(lexicon.items(), key=lambda kv: -kv[1][1])[:8]
print("Most common  :", [(w, info[1]) for w, info in top])

# ids (first number) must be exactly 0 to n-1 with no gaps
ids = sorted(info[0] for info in lexicon.values())
print("IDs 0..n-1   :", ids == list(range(len(lexicon))))

Lexicon loaded in 0.1s
Unique words : 66,288
Memory       : 331 MB now (+4 MB for the lexicon)
File         : /content/drive/MyDrive/DSA_Search_Engine/index_data/lexicon.json (1.51 MB)
First 3      : {'0': [0, 1115], "0'": [1, 6], "0'10": [2, 1]}
Most common  : [('the', 54865), ('and', 54053), ('to', 53788), ('a', 53703), ('it', 51369), ('i', 50936), ('is', 50619), ('of', 49702)]
IDs 0..n-1   : True


In [ ]:
import os


from src.lexicon import build_lexicon as module_build_lexicon

# Use a temporary file for testing
TEST_LEXICON_PATH = '/content/test_lexicon.json'

# Build using the original notebook function
original_lexicon = build_lexicon(
    SAMPLE_PATH,
    TEST_LEXICON_PATH
)

# Build using the new module
module_lexicon = module_build_lexicon(
    SAMPLE_PATH,
    TEST_LEXICON_PATH
)

# Compare both results
print("Original unique words:", len(original_lexicon))
print("Module unique words:  ", len(module_lexicon))

print("First 3 original:", list(original_lexicon.items())[:3])
print("First 3 module:  ", list(module_lexicon.items())[:3])

print("Lexicons match:", original_lexicon == module_lexicon)

assert original_lexicon == module_lexicon

print("\nSUCCESS: Module Lexicon matches the original implementation.")

# Remove the temporary test file
os.remove(TEST_LEXICON_PATH)

Original unique words: 6998
Module unique words:   6998
First 3 original: [('0', [0, 8]), ('000', [1, 3]), ('01', [2, 1])]
First 3 module:   [('0', [0, 8]), ('000', [1, 3]), ('01', [2, 1])]
Lexicons match: True

SUCCESS: Module Lexicon matches the original implementation.


In [ ]:
import os

from src.lexicon import (
    build_lexicon as module_build_lexicon,
    load_lexicon as module_load_lexicon
)

# Create a temporary Lexicon file for testing
TEST_LEXICON_PATH = '/content/test_lexicon.json'

# Build and save the Lexicon using the new module
module_lexicon = module_build_lexicon(
    SAMPLE_PATH,
    TEST_LEXICON_PATH
)

# Load the same file using both implementations
original_loaded = load_lexicon(TEST_LEXICON_PATH)
module_loaded = module_load_lexicon(TEST_LEXICON_PATH)

# Compare the results
print("Original loaded words:", len(original_loaded))
print("Module loaded words:  ", len(module_loaded))

print("Both loaded Lexicons match:",
      original_loaded == module_loaded)

print("Loaded Lexicon matches built Lexicon:",
      module_loaded == module_lexicon)

assert original_loaded == module_loaded
assert module_loaded == module_lexicon

# Remove the temporary file
os.remove(TEST_LEXICON_PATH)

print("\nSUCCESS: Module load_lexicon() works correctly.")

Original loaded words: 6998
Module loaded words:   6998
Both loaded Lexicons match: True
Loaded Lexicon matches built Lexicon: True

SUCCESS: Module load_lexicon() works correctly.


In [ ]:
# Cell 2D: Build or load the Forward Index

# Import the functions from our Forward Index module.
# We will create these functions in src/forward_index.py.
from src.forward_index import (
    build_forward_index,
    forward_index_is_current,
    load_meta,
    iter_forward_index
)

# Record the memory before working on the Forward Index.
# We will compare it with the memory used afterwards.
start_mem = memory_mb()

# Start the timer to measure how long this cell takes.
t0 = time.time()


# Check whether the saved Forward Index is still up to date.
# This checks the dataset and Lexicon used to build it.
current, reason = forward_index_is_current(
    SOURCE_PATH,
    LEXICON_PATH,
    FORWARD_PATH,
    META_PATH
)


# If the saved index is current and we do not want to rebuild,
# simply load its metadata instead of building it again.
if current and not REBUILD:

    meta = load_meta(META_PATH)
    action = "found, up to date"

else:

    # If the existing index is outdated, show why it needs rebuilding.
    if not current:
        print("Rebuilding:", reason)

    # Build the Forward Index using:
    # 1. The selected dataset.
    # 2. The Lexicon already loaded in memory.
    # 3. The paths where the Lexicon, Forward Index and metadata are saved.
    meta = build_forward_index(
        SOURCE_PATH,
        lexicon,
        LEXICON_PATH,
        FORWARD_PATH,
        META_PATH
    )

    action = "built"


# Show how long the loading or building process took.
print(f"Forward index {action} in {time.time() - t0:.1f}s")

# Show basic information about the Forward Index.
print(f"Reviews        : {meta['num_docs']:,}")
print(f"Total tokens   : {meta['total_tokens']:,}")
print(f"Average length : {meta['avg_doc_len']:.0f} words")
print(f"Vocabulary     : {meta['vocab_size']:,} words")

# Show current memory usage and how much it changed.
print(f"Memory         : {memory_mb():.0f} MB now (+{memory_mb() - start_mem:.0f} MB)")

# Show the size of the saved Forward Index file.
print(f"File           : {FORWARD_PATH} ({os.path.getsize(FORWARD_PATH) / 1e6:.2f} MB)")


# Create a reverse Lexicon so we can convert word IDs back into words.
# This is only for displaying and checking the index.
id_to_word = {
    info[0]: word
    for word, info in lexicon.items()
}

# Read the first review from the saved Forward Index.
# We use next() because we only want to inspect one review here.
first = next(iter_forward_index(FORWARD_PATH))

# Display some basic information about the first review.
print(
    "First review   :",
    first["doc_id"],
    first["rating"],
    first["title"][:50]
)

# Convert the first five word IDs back into words.
# Each term contains: word ID, term frequency, title presence and first position.
print(
    "First 5 terms  :",
    [
        (id_to_word[w], tf, t, p)
        for w, tf, t, p in first["terms"][:5]
    ]
)

Rebuilding: no saved forward index
Forward index built in 0.2s
Reviews        : 400
Total tokens   : 112,101
Average length : 280 words
Vocabulary     : 6,998 words
Memory         : 332 MB now (+0 MB)
File           : /content/drive/MyDrive/DSA_Search_Engine/index_data/sample/forward_index.jsonl (0.92 MB)
First review   : 0 5.0 if its prime day and you're contemplating, just st
First 5 terms  : [('1', 1, 0, 40), ('3', 2, 0, 111), ('7', 1, 0, 39), ('a', 2, 0, 20), ('about', 3, 0, 7)]


In [ ]:
%%writefile /content/drive/MyDrive/DSA_Search_Engine/src/forward_index.py

# Forward Index module

Writing /content/drive/MyDrive/DSA_Search_Engine/src/forward_index.py


In [ ]:
from src.forward_index import (
    build_forward_index,
    forward_index_is_current,
    load_meta,
    iter_forward_index
)

print("Forward Index module imported successfully!")

Forward Index module imported successfully!


In [ ]:
from src.forward_index import make_forward_entry

# Get one review from the sample dataset.
record = next(iter_records(SAMPLE_PATH))

# Create its Forward Index entry.
entry = make_forward_entry(record, lexicon)

# Display the result.
print("Document ID:", entry["doc_id"])
print("Title:", entry["title"])
print("Document length:", entry["doc_len"])
print("Number of unique terms:", len(entry["terms"]))
print("First 5 terms:", entry["terms"][:5])

Document ID: 0
Title: if its prime day and you're contemplating, just stop -- do it.
Document length: 182
Number of unique terms: 147
First 5 terms: [[245, 1, 0, 40], [1923, 2, 0, 111], [3403, 1, 0, 39], [4111, 2, 0, 20], [4351, 3, 0, 7]]


In [ ]:
# Check whether the Forward Index is up to date.
current, reason = forward_index_is_current(
    SOURCE_PATH,
    LEXICON_PATH,
    FORWARD_PATH,
    META_PATH
)

print("Is current:", current)
print("Reason:", reason)

Is current: True
Reason: up to date


In [ ]:
# Load the sample Lexicon.
sample_lexicon = load_lexicon(
    os.path.join(INDEX_DIR, 'sample', 'lexicon.json')
)

print("Sample Lexicon loaded!")
print("Unique words:", len(sample_lexicon))

Sample Lexicon loaded!
Unique words: 6998


In [ ]:
# Use the sample Lexicon for this test.
lexicon = sample_lexicon

print("Using sample dataset:", SOURCE_PATH)
print("Using sample Lexicon:", len(lexicon), "words")
print("Forward Index will be saved at:", FORWARD_PATH)

Using sample dataset: /content/drive/MyDrive/DSA_Search_Engine/sample_dataset.jsonl
Using sample Lexicon: 6998 words
Forward Index will be saved at: /content/drive/MyDrive/DSA_Search_Engine/index_data/sample/forward_index.jsonl


In [ ]:
current, reason = forward_index_is_current(
    SOURCE_PATH,
    LEXICON_PATH,
    FORWARD_PATH,
    META_PATH
)

sample_meta = load_meta(META_PATH)

print("Is current:", current)
print("Reason:", reason)
print("Reviews:", sample_meta["num_docs"])
print("Vocabulary:", sample_meta["vocab_size"])

Is current: True
Reason: up to date
Reviews: 400
Vocabulary: 6998
